<h2> RAG Evaluation </h2>

RAG evaluation checks the whole flow together.

This includes:
search
prompt
LLM

In this part, we'll evaluate:

RAG answers with an LLM judge
Agent answers and tool-call trajectories

<h3> LLM as a judge </h3>
For RAG and agent evaluation, we compare the generated answer with the original answer. The generated answer won't use the same words as the original. It's a generative model, so the phrasing will be different even when the meaning is the same.

This is why we use another LLM to do the comparison. We show the judge the question, the original answer, and the generated answer. Then we ask it to decide if they are semantically equivalent.

This approach is called LLM-as-a-judge. The evaluating LLM is the judge. It classifies each answer as good or bad and explains its reasoning. Asking the judge to explain why it made a decision generally produces better classifications than asking for just the verdict.

<h3> How to Evaluate </h3>
For each generated question, we run RAG and save the answer produced by the LLM. Later, we'll compare this answer with the original FAQ answer.

This is the A->Q->A' setup:

A = original answer in the FAQ
Q = generated question from this answer
A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our questions came from FAQ records. For each question, we know which original answer it came from.


In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# Create a lookup table for the original FAQ documents:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

# We'll use this lookup table to find the original answer for each ground truth question.

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

use RAGWithUsage from the evaluation utilities. It subclasses RAGBase from module 01, so it has the same rag method.

It stores token usage after each LLM call. Then we can calculate the total cost later.

It also uses the search boosts we selected in the search tuning lesson: question=1.0, answer=2.0, and section=0.1.

In [7]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, RAGBase searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. We save the answer so the next lesson can judge it.

In [8]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course if you found it late.\n\nIf you want to receive a certificate, though, you need to submit your project while the course is still accepting submissions.'

question
   ↓
comes from ground_truth-new.csv

assistant.rag(question)
   ↓
uses the question to search documents

answer_llm
   ↓
is generated by the LLM

In [9]:
assistant.total_cost()

0.0005685

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [11]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late.\n\nIf you want to receive a certificate, though, you need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late.\n\nHowever, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
assistant.reset_usage()

In [15]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Run RAG for all ground truth questions:

In [17]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

KeyboardInterrupt: 

generate_rag_answer returns one answer record for each question.

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
assistant.total_cost()

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)